# Predictive Maintenance for Power Generation Assets - Synthetic Data Generator

## Overview
This notebook generates synthetic data simulating power generation asset monitoring:
- **Vibration sensors** - acceleration data from turbines and generators
- **Thermal data** - temperature readings from bearings, windings
- **Oil analysis** - dissolved gas and particle counts
- **Failure events** - historical maintenance and failure records

## Tables Created
| Table | Description | Volume |
|-------|-------------|--------|
| `dim_assets` | Power generation assets | ~100 rows |
| `dim_sensors` | Sensor registry | ~500 rows |
| `fact_vibration` | Vibration readings | 500K+ rows/day |
| `fact_thermal` | Temperature readings | 100K+ rows/day |
| `fact_oil_analysis` | Oil sample results | ~500 rows/month |
| `fact_maintenance_events` | Work orders and failures | ~1K rows/year |

## How to Run in Fabric
1. Attach this notebook to a Lakehouse
2. Run all cells sequentially
3. Data will be written as Delta tables

In [ ]:
!pip install faker

In [ ]:
# Configuration
SEED = 42
NUM_PLANTS = 5
ASSETS_PER_PLANT = 20
SENSORS_PER_ASSET = 5

from datetime import datetime, timedelta
START_DATE = datetime(2024, 1, 1)
END_DATE = datetime(2024, 1, 7)

In [ ]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random
import uuid

np.random.seed(SEED)
random.seed(SEED)
fake = Faker()
Faker.seed(SEED)

print("Libraries loaded successfully")

## 1. Generate Asset Dimension

In [ ]:
def generate_assets(num_plants, assets_per_plant):
    asset_types = [
        ('Gas Turbine', 150, 25000),
        ('Steam Turbine', 200, 30000),
        ('Generator', 300, 50000),
        ('Transformer', 150, 20000),
        ('Compressor', 100, 15000),
        ('Pump', 50, 5000),
        ('Fan', 30, 3000),
        ('Motor', 75, 8000)
    ]
    
    criticality_levels = ['Critical', 'High', 'Medium', 'Low']
    
    assets = []
    for plant_idx in range(num_plants):
        plant_id = f'PLANT-{str(plant_idx+1).zfill(3)}'
        plant_name = f'{fake.city()} Power Station'
        
        for asset_idx in range(assets_per_plant):
            asset_type, rated_power, rpm = random.choice(asset_types)
            install_date = fake.date_between(start_date='-25y', end_date='-1y')
            
            assets.append({
                'asset_id': f'{plant_id}-{asset_type[:3].upper()}-{str(asset_idx+1).zfill(3)}',
                'plant_id': plant_id,
                'plant_name': plant_name,
                'asset_type': asset_type,
                'manufacturer': random.choice(['GE', 'Siemens', 'Mitsubishi', 'ABB', 'Hitachi']),
                'model': f'{fake.lexify(text="????").upper()}-{random.randint(1000,9999)}',
                'serial_number': fake.uuid4()[:12].upper(),
                'install_date': install_date,
                'rated_power_mw': rated_power + random.randint(-20, 50),
                'rated_rpm': rpm + random.randint(-500, 500),
                'criticality': random.choices(criticality_levels, weights=[0.2, 0.3, 0.35, 0.15])[0],
                'operating_hours': random.randint(10000, 150000),
                'last_major_overhaul': fake.date_between(start_date=install_date, end_date='today')
            })
    
    return pd.DataFrame(assets)

df_assets = generate_assets(NUM_PLANTS, ASSETS_PER_PLANT)
print(f"Generated {len(df_assets)} assets")
df_assets.head()

In [ ]:
def generate_sensors(assets_df, sensors_per_asset):
    sensor_types = [
        ('Accelerometer', 'mm/s', 'Vibration'),
        ('Temperature', '°C', 'Thermal'),
        ('Pressure', 'bar', 'Process'),
        ('Speed', 'RPM', 'Rotation'),
        ('Current', 'A', 'Electrical')
    ]
    
    locations = ['Drive End Bearing', 'Non-Drive End Bearing', 'Gearbox', 'Casing', 'Inlet', 'Outlet']
    
    sensors = []
    for _, asset in assets_df.iterrows():
        for i in range(sensors_per_asset):
            sensor_type, unit, category = random.choice(sensor_types)
            sensors.append({
                'sensor_id': f'{asset["asset_id"]}-SEN-{str(i+1).zfill(2)}',
                'asset_id': asset['asset_id'],
                'sensor_type': sensor_type,
                'measurement_unit': unit,
                'category': category,
                'location': random.choice(locations),
                'sampling_rate_hz': random.choice([1, 10, 100, 1000]),
                'install_date': fake.date_between(start_date='-5y', end_date='-1m'),
                'calibration_due': fake.date_between(start_date='today', end_date='+1y')
            })
    
    return pd.DataFrame(sensors)

df_sensors = generate_sensors(df_assets, SENSORS_PER_ASSET)
print(f"Generated {len(df_sensors)} sensors")
df_sensors.head()

## 2. Generate Vibration Data

In [ ]:
def generate_vibration_data(sensors_df, assets_df, start_time, hours=1):
    """Generate vibration readings with degradation patterns"""
    
    vibration_sensors = sensors_df[sensors_df['sensor_type'] == 'Accelerometer'].copy()
    readings = []
    
    for _, sensor in vibration_sensors.iterrows():
        asset = assets_df[assets_df['asset_id'] == sensor['asset_id']].iloc[0]
        
        # Base vibration level depends on asset age and operating hours
        age_years = (datetime.now() - pd.to_datetime(asset['install_date'])).days / 365
        degradation_factor = 1 + (age_years / 25) * 0.5 + (asset['operating_hours'] / 150000) * 0.3
        
        # Simulate readings every 10 minutes
        current_time = start_time
        while current_time < start_time + timedelta(hours=hours):
            # Add random spikes for developing faults
            is_fault_developing = random.random() < 0.05
            
            base_velocity = 2.0 * degradation_factor  # mm/s
            if is_fault_developing:
                base_velocity *= random.uniform(1.5, 3.0)
            
            readings.append({
                'reading_id': str(uuid.uuid4()),
                'sensor_id': sensor['sensor_id'],
                'asset_id': sensor['asset_id'],
                'timestamp': current_time,
                'velocity_mm_s': round(base_velocity + np.random.normal(0, 0.3), 3),
                'acceleration_g': round((base_velocity / 10) + np.random.normal(0, 0.05), 4),
                'displacement_um': round(base_velocity * 5 + np.random.normal(0, 2), 2),
                'peak_frequency_hz': round(asset['rated_rpm'] / 60 + np.random.normal(0, 1), 2),
                'rms_value': round(base_velocity * 0.707 + np.random.normal(0, 0.1), 3),
                'fault_indicator': is_fault_developing
            })
            
            current_time += timedelta(minutes=10)
    
    return pd.DataFrame(readings)

df_vibration = generate_vibration_data(df_sensors, df_assets, START_DATE, hours=24)
print(f"Generated {len(df_vibration)} vibration readings")
print(f"Fault indicators: {df_vibration['fault_indicator'].sum()}")
df_vibration.head()

## 3. Generate Thermal Data

In [ ]:
def generate_thermal_data(sensors_df, assets_df, start_time, hours=1):
    """Generate temperature readings with thermal patterns"""
    
    temp_sensors = sensors_df[sensors_df['sensor_type'] == 'Temperature'].copy()
    readings = []
    
    for _, sensor in temp_sensors.iterrows():
        asset = assets_df[assets_df['asset_id'] == sensor['asset_id']].iloc[0]
        
        # Base temperature depends on location
        location_temps = {
            'Drive End Bearing': 65,
            'Non-Drive End Bearing': 60,
            'Gearbox': 75,
            'Casing': 55,
            'Inlet': 40,
            'Outlet': 85
        }
        base_temp = location_temps.get(sensor['location'], 60)
        
        # Degradation increases temperature
        operating_hours = asset['operating_hours']
        degradation_temp = (operating_hours / 150000) * 10
        
        current_time = start_time
        while current_time < start_time + timedelta(hours=hours):
            # Add daily load variation
            hour = current_time.hour
            load_factor = 0.8 + 0.2 * np.sin((hour - 6) * np.pi / 12) if 6 <= hour <= 22 else 0.7
            
            temperature = base_temp * load_factor + degradation_temp + np.random.normal(0, 2)
            
            # Flag high temperatures
            is_high_temp = temperature > (base_temp * 1.3)
            
            readings.append({
                'reading_id': str(uuid.uuid4()),
                'sensor_id': sensor['sensor_id'],
                'asset_id': sensor['asset_id'],
                'timestamp': current_time,
                'temperature_c': round(temperature, 1),
                'ambient_temp_c': round(25 + np.random.normal(0, 3), 1),
                'delta_t': round(temperature - 25, 1),
                'high_temp_alarm': is_high_temp
            })
            
            current_time += timedelta(minutes=5)
    
    return pd.DataFrame(readings)

df_thermal = generate_thermal_data(df_sensors, df_assets, START_DATE, hours=24)
print(f"Generated {len(df_thermal)} thermal readings")
print(f"High temp alarms: {df_thermal['high_temp_alarm'].sum()}")
df_thermal.head()

## 4. Generate Oil Analysis Data

In [ ]:
def generate_oil_analysis(assets_df, num_samples_per_asset=3):
    """Generate oil analysis results"""
    
    # Filter to rotating equipment that uses oil
    oil_assets = assets_df[assets_df['asset_type'].isin(['Gas Turbine', 'Steam Turbine', 'Generator', 'Compressor'])]
    
    samples = []
    for _, asset in oil_assets.iterrows():
        for i in range(num_samples_per_asset):
            sample_date = fake.date_between(start_date='-6m', end_date='today')
            
            # Generate dissolved gas concentrations (ppm)
            # Normal ranges vary by gas
            samples.append({
                'sample_id': str(uuid.uuid4()),
                'asset_id': asset['asset_id'],
                'sample_date': sample_date,
                'hydrogen_ppm': round(max(0, np.random.lognormal(2, 1)), 1),
                'methane_ppm': round(max(0, np.random.lognormal(1.5, 0.8)), 1),
                'ethane_ppm': round(max(0, np.random.lognormal(1, 0.7)), 1),
                'ethylene_ppm': round(max(0, np.random.lognormal(0.5, 0.5)), 1),
                'acetylene_ppm': round(max(0, np.random.exponential(0.5)), 2),
                'carbon_monoxide_ppm': round(max(0, np.random.lognormal(2, 0.8)), 1),
                'carbon_dioxide_ppm': round(max(0, np.random.lognormal(5, 1)), 1),
                'moisture_ppm': round(max(0, np.random.lognormal(2.5, 0.5)), 1),
                'particle_count_4um': random.randint(100, 5000),
                'particle_count_14um': random.randint(10, 500),
                'viscosity_cst': round(32 + np.random.normal(0, 2), 1),
                'acid_number': round(max(0, np.random.exponential(0.1)), 3),
                'oil_condition': random.choices(['Good', 'Monitor', 'Alert', 'Critical'], 
                                               weights=[0.6, 0.25, 0.12, 0.03])[0]
            })
    
    return pd.DataFrame(samples)

df_oil = generate_oil_analysis(df_assets)
print(f"Generated {len(df_oil)} oil analysis samples")
print(f"Condition breakdown: {df_oil['oil_condition'].value_counts().to_dict()}")
df_oil.head()

## 5. Generate Maintenance Events

In [ ]:
def generate_maintenance_events(assets_df, events_per_asset=5):
    """Generate historical maintenance and failure records"""
    
    event_types = [
        ('Planned Maintenance', 'Scheduled', 0.5),
        ('Corrective Maintenance', 'Unplanned', 0.25),
        ('Failure', 'Emergency', 0.1),
        ('Inspection', 'Scheduled', 0.1),
        ('Calibration', 'Scheduled', 0.05)
    ]
    
    failure_modes = [
        'Bearing Failure', 'Seal Leak', 'Vibration Excessive',
        'Overheating', 'Electrical Fault', 'Lubrication Issue',
        'Imbalance', 'Misalignment', 'Blade Damage', 'Corrosion'
    ]
    
    events = []
    for _, asset in assets_df.iterrows():
        num_events = random.randint(1, events_per_asset)
        
        for _ in range(num_events):
            event_type, category, _ = random.choices(event_types, weights=[e[2] for e in event_types])[0]
            event_date = fake.date_time_between(start_date='-3y', end_date='now')
            
            downtime_hours = 0
            failure_mode = None
            cost = 0
            
            if event_type == 'Failure':
                downtime_hours = random.randint(24, 720)  # 1 to 30 days
                failure_mode = random.choice(failure_modes)
                cost = random.randint(50000, 2000000)
            elif event_type == 'Corrective Maintenance':
                downtime_hours = random.randint(4, 72)
                failure_mode = random.choice(failure_modes)
                cost = random.randint(5000, 100000)
            elif event_type == 'Planned Maintenance':
                downtime_hours = random.randint(8, 168)
                cost = random.randint(10000, 500000)
            
            events.append({
                'event_id': str(uuid.uuid4()),
                'asset_id': asset['asset_id'],
                'event_type': event_type,
                'category': category,
                'event_datetime': event_date,
                'completion_datetime': event_date + timedelta(hours=downtime_hours) if downtime_hours > 0 else event_date,
                'downtime_hours': downtime_hours,
                'failure_mode': failure_mode,
                'cost_usd': cost,
                'work_order_number': f'WO-{fake.random_number(digits=8)}',
                'technician': fake.name(),
                'notes': fake.sentence() if random.random() > 0.5 else None
            })
    
    return pd.DataFrame(events)

df_maintenance = generate_maintenance_events(df_assets)
print(f"Generated {len(df_maintenance)} maintenance events")
print(f"Event types: {df_maintenance['event_type'].value_counts().to_dict()}")
df_maintenance.head()

## 6. Data Validation

In [ ]:
print("=== Data Validation Report ===")
print(f"\nAssets: {len(df_assets)} records")
print(f"  - By type: {df_assets['asset_type'].value_counts().to_dict()}")
print(f"  - By criticality: {df_assets['criticality'].value_counts().to_dict()}")

print(f"\nSensors: {len(df_sensors)} records")
print(f"  - Valid FK: {df_sensors['asset_id'].isin(df_assets['asset_id']).all()}")

print(f"\nVibration: {len(df_vibration)} records")
print(f"  - Velocity range: {df_vibration['velocity_mm_s'].min():.2f} - {df_vibration['velocity_mm_s'].max():.2f} mm/s")

print(f"\nThermal: {len(df_thermal)} records")
print(f"  - Temp range: {df_thermal['temperature_c'].min():.1f} - {df_thermal['temperature_c'].max():.1f} °C")

print(f"\nOil Analysis: {len(df_oil)} records")
print(f"\nMaintenance: {len(df_maintenance)} records")
print(f"  - Total failures: {len(df_maintenance[df_maintenance['event_type'] == 'Failure'])}")
print(f"  - Total downtime: {df_maintenance['downtime_hours'].sum():,.0f} hours")
print(f"  - Total cost: ${df_maintenance['cost_usd'].sum():,.0f}")

## 7. Write to Lakehouse

In [ ]:
# Uncomment for Fabric execution
# from pyspark.sql import SparkSession
# spark = SparkSession.builder.getOrCreate()

# spark.createDataFrame(df_assets).write.mode("overwrite").format("delta").saveAsTable("dim_assets")
# spark.createDataFrame(df_sensors).write.mode("overwrite").format("delta").saveAsTable("dim_sensors")
# spark.createDataFrame(df_vibration).write.mode("append").format("delta").saveAsTable("fact_vibration")
# spark.createDataFrame(df_thermal).write.mode("append").format("delta").saveAsTable("fact_thermal")
# spark.createDataFrame(df_oil).write.mode("append").format("delta").saveAsTable("fact_oil_analysis")
# spark.createDataFrame(df_maintenance).write.mode("append").format("delta").saveAsTable("fact_maintenance_events")

# Local testing
df_assets.to_csv('dim_assets.csv', index=False)
df_vibration.to_csv('fact_vibration.csv', index=False)
print("Data saved successfully")

## Known Limitations
1. Vibration patterns are simplified - real FFT spectra have complex harmonic content
2. Degradation is linear - actual degradation follows various curves
3. No correlation between failures and sensor readings in training data

## Next Steps
1. Connect to Digital Twin Builder for asset hierarchy
2. Train RUL prediction model on vibration trends
3. Build failure mode classification from maintenance records